# 实验六：使用卷积神经网络识别 CIFAR-10 彩色图像

第五次实验中，我们把手写数字展平后交给全连接网络。这次实验，我们将研究更加丰富多样的图像，探究 CNN 在图片颜色、位置上的处理。

这次实验仍使用 PyTorch 完成。代码中的工具函数已给出，只需完成代码中的相应部分即可。同学们在完成实验的过程中，尤其要注意以下这些内容：

1. **RGB通道**：与上次处理的灰度图不同，我们这次处理的是以张量形式存储的图片。对于这种包含更多信息的数据形式，我们又怎么进行处理？
2. **卷积核的作用**：CNN 网络最核心的一个内容就是卷积核，同学们在处理的过程中，要感受卷积核对图像局部信息提取的作用。同时也应注意到，卷积核对信息的提取与图像信息的位置无关。
3. **模型比较**：实验的最后会进行与上次实验的比较。虽然处理的是不同数据集，但在这种更复杂的数据集上， CNN 也能展示其强大之处。感受 CNN 与上次全连接网络的不同与更先进的地方。

实验主线：认识彩图 → 理解卷积 → 搭建 CNN → 检查与训练 → 比较模型 → 数据增强对照 → 最终测试与错误分析。


## 0 · 环境、文件与实验约定

依然使用之前使用的 `ai-lab` 虚拟环境即可。

由于这里的数据集较大，助教会提供两份压缩材料，一份提供数据一份不提供。如果下载不包含数据集的材料，第一次运行时将 `DOWNLOAD=True`；缓存完整后改回 `False`。

| 项目               | 本实验约定                                                                                  |
| ------------------ | ------------------------------------------------------------------------------------------- |
| 数据划分           | 官方训练部分 50,000 张固定拆成 45,000 张训练池、5,000 张验证；官方 10,000 张测试图像留到 Q7 |
| quick 模式（默认） | 从训练池固定抽取 12,000 张，验证仍为全部 5,000 张；训练 8 轮                                |
| full 模式          | 使用全部 45,000 张训练图像，验证仍为 5,000 张；训练 12 轮                                   |
| 预处理             | 像素先转 float32 的 [0,1]，再逐通道用 `(x−0.5)/0.5` 映射到 [−1,1]                           |
| 优化               | Adam，lr=0.001，batch size=128；三组主实验保持一致                                          |
| 输出与损失         | 输出十个 logits；使用 CrossEntropyLoss，不提前加 softmax                                    |
| 选模               | 验证准确率高者优先，持平时验证损失小者优先，再持平取较早轮次                                |
| 设备               | CPU即可，如果有 GPU 或 MPS 请自行查阅资料并移动数据集至相应设备上计算                       |

说明：

- quick 与 full 是两套不同的实验预算，用于区分“充分训练”和“不充分”训练的两种情况；
- Adam 是根据梯度更新参数的另一种优化器，直接使用即可。


In [ ]:
import os
import platform
import sys
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from IPython.display import Markdown, display
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset

assert Path("lab_utils.py").exists(), "请在实验六目录运行 notebook"
from lab_utils import (
    CLASS_NAMES_ZH,
    CLASSES,
    data_root,
    display_image,
    evaluate,
    fit_model,
    load_test,
    load_train_val,
    make_loader,
    plot_channels,
    plot_confusion,
    plot_history,
    plot_samples,
    print_comparison,
    seed_everything,
    snapshot,
    with_augmentation,
)

SEED, SPLIT_SEED, ORDER_SEED = 42, 42, 2026
MODE = os.environ.get("LAB6_MODE", "quick")  # 改为 "full" 使用完整训练池。
SETTINGS = {
    "quick": {"train_limit": 12000, "epochs": 8},
    "full": {"train_limit": None, "epochs": 12},
}
assert MODE in SETTINGS, "MODE 应为 quick 或 full"
TRAIN_LIMIT, EPOCHS = SETTINGS[MODE]["train_limit"], SETTINGS[MODE]["epochs"]
BATCH_SIZE, LR = 128, 0.001
DOWNLOAD = False
DEVICE = torch.device(os.environ.get("LAB6_DEVICE", "cpu"))
torch.set_num_threads(min(4, os.cpu_count() or 1))
seed_everything(SEED)

print(
    "Python:",
    sys.version.split()[0],
    "PyTorch:",
    torch.__version__,
    "torchvision:",
    torchvision.__version__,
)
print(
    "device:",
    DEVICE,
    "machine:",
    platform.machine(),
)
print("mode:", MODE, "epochs:", EPOCHS, "batch size:", BATCH_SIZE, "lr:", LR)
print("data root:", data_root().resolve())

# CLASSES 和 CLASS_NAMES_ZH 是本次实验区分的目标，已经在工具模块中定义好
print(dict(zip(CLASSES, CLASS_NAMES_ZH)))


## Q1 · 彩色图像处理入门

对于一张彩色图像，有 R、G、B 三个颜色通道。相比于上次的灰度图，彩图的三种颜色是存放在 `Channel` 这个维度上的，整体上就是 $(\text{Height}, \text{Width}, \text{Channel})$。所谓 Channel，直观上理解就是几个平行的、完全相同的矩阵堆叠在一起。值得注意的是， PyTorch 卷积层使用的是 $(\text{Batch}, \text{Channel}, \text{Height}, \text{Width})$。这里的 Batch 顾名思义，就是这一批图像中有多少图像。

下面的第一段代码为数据加载与检查，确保数据集的完整性后再补全下面的代码块。


In [ ]:
data = load_train_val(download=DOWNLOAD, train_limit=TRAIN_LIMIT, split_seed=SPLIT_SEED)
train_set, train_eval_set, val_set = data["train"], data["train_eval"], data["val"]
train_indices, val_indices = set(train_set.indices), set(val_set.indices)
assert train_indices.isdisjoint(val_indices)
assert train_set.indices == train_eval_set.indices
assert len(val_set) == 5000

x_batch, y_batch = next(iter(make_loader(train_eval_set, BATCH_SIZE)))
print("train/validation:", len(train_set), len(val_set))
print(
    "image:",
    tuple(x_batch.shape),
    x_batch.dtype,
    "range:",
    (float(x_batch.min()), float(x_batch.max())),
)
print("label:", tuple(y_batch.shape), y_batch.dtype)
counts = np.bincount(
    [train_set.base.targets[i] for i in train_set.indices], minlength=10
)
print("training class counts:", dict(zip(CLASSES, counts.tolist())))
_ = plot_samples(x_batch[:12], y_batch[:12], title="Training samples")


下面这一段代码展示的是，对一张你 RGB 图像，我们如何按三个通道进行拆分。对于单个通道，我们使用灰度图来进行展示。亮处表示该处颜色分量的数值较大。这与我们将黑白格子映射成一个 $\mathbb{R}$ 上的矩阵是完全同理的。

> 这里可以通过修改 `x_batch[]` 的下标来选择自己想拆分的不同的图像。


In [ ]:
image = x_batch[7]
rgb = display_image(image)
# 反标准化，并将 CHW 转为 HWC，仅用于绘图。
# 这里的 `rgb` 是一个三维数组
# TODO：从 rgb 数组中取出红、绿、蓝三个通道。
red = ...
green = ...
blue = ...
assert rgb.shape == (32, 32, 3)
assert red.shape == green.shape == blue.shape == (32, 32)
_ = plot_channels(image)
print("single image:", tuple(image.shape), "flattened:", image.numel())


## Q2 · 卷积核工作原理

课上我们已经讲述了卷积核的工作原理。这里我们进行一个简单的手算，复习相关原理。

先看单输入、单输出通道的小矩阵。让核 `[[1,0],[0,-1]]` 在图像上滑动，每个位置做逐元素乘积再求和；不填充，步幅为 1，无偏置。

输入为 `[[1,2,3],[4,5,6],[7,8,9]]`。请先口算 / 用代码表达式直接计算左上角的输出，并预测完整输出形状，再用代码核验。PyTorch 的卷积层不翻转核，严格说是互相关运算；深度学习中通常也称作卷积。


In [ ]:
small_x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]])[None, None]
"""定义的矩阵"""
kernel = torch.tensor([[1.0, 0.0], [0.0, -1.0]])[None, None]
"""定义的卷积核"""

top_left = ...  # TODO：写出左上角位置的乘积与求和。不想口算可以用表达式写出来计算

conv_out = F.conv2d(...)
# TODO：用 F.conv2d 进行计算，查阅其用法与字段
# HINT：输入 input 是什么？weight 使用了什么卷积核？
# HINT: stride=1 步长、padding=0 填充

pool_out = F.max_pool2d(...)
# TODO：用 F.max_pool2d，kernel_size=2、stride=1。
# HINT：用法及参数与上基本相同

print("hand calculation:", top_left)
print("convolution:\n", conv_out[0, 0])
print("max pooling:\n", pool_out[0, 0])

# 手算后的结果。
assert tuple(conv_out.shape) == (1, 1, 2, 2)
assert torch.allclose(conv_out, torch.full((1, 1, 2, 2), -4.0))
assert torch.equal(pool_out[0, 0], torch.tensor([[5.0, 6.0], [8.0, 9.0]]))


下面是一个助教提供的固定边缘滤波器。将这个滤波器作用在我们的图像上，观察其特点，为什么能称之为“边缘滤波器”。这个例子也很好地说明了卷积核对图像特征提取的作用。


In [ ]:
gray = torch.tensor(rgb.mean(axis=2), dtype=torch.float32)[None, None]
edge_kernel = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]])[
    None, None
]
edges = F.conv2d(gray, edge_kernel, padding=1)[0, 0].numpy()
scale = max(float(np.abs(edges).max()), 1e-6)
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(rgb)
axes[0].set_title("RGB input")
axes[1].imshow(gray[0, 0], cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Mean of channels")
axes[2].imshow(edges, cmap="coolwarm", vmin=-scale, vmax=scale)
axes[2].set_title("Fixed edge filter")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()


## Q3 · 搭建全连接基线与小型 CNN

使用相同的 RGB 输入，建立两种模型：

- MLP：`Flatten → Linear(3072,128) → ReLU → Linear(128,10)`。
- CNN：按下表连接。卷积使用默认偏置，最后直接输出 logits。

| 层            | 设置                              | 输出形状（待验证） |
| ------------- | --------------------------------- | ------------------ |
| 输入          | RGB，32×32                        | `(B,3,32,32)`      |
| 卷积1 + ReLU  | 3→16，核3×3，padding=1，stride=1  | `(B,16,32,32)`     |
| 最大池化1     | 核2×2，stride=2                   | `(B,16,16,16)`     |
| 卷积2 + ReLU  | 16→32，核3×3，padding=1，stride=1 | `(B,32,16,16)`     |
| 最大池化2     | 核2×2，stride=2                   | `(B,32,8,8)`       |
| Flatten       | 保留 batch 维度                   | `(B,2048)`         |
| 全连接 + ReLU | 2048→64                           | `(B,64)`           |
| 输出          | 64→10                             | `(B,10)`           |

卷积层参数量为 `输出通道数 × (输入通道数 × 核高 × 核宽 + 1)`，其中 `+1` 对应偏置。空间大小可用 `floor((输入大小+2×padding−核大小)/stride)+1` 计算（本次 dilation=1）。


In [ ]:
def build_model(kind: Literal["cnn", "mlp"] = "cnn", seed=42):
    # 这里的 Literal 是多选一的意思，默认为 cnn，但同学们还是要按分支实现不同内容
    # 设定随机种子
    seed_everything(seed)
    # TODO：按题目完成 mlp 和 cnn；用 nn.Sequential 连接各层。
    # 提示：卷积层为 nn.Conv2d，池化为 nn.MaxPool2d，激活为 nn.ReLU
    # TODO：未知 kind 抛出 ValueError；将模型移动到 DEVICE。
    raise NotImplementedError("Q3：完成两种模型")


上面代码块填充完毕后运行下方代码，确保参数量是对应的，不要缺漏某一部分。从形式上也能看到，每一层的输入、输出是怎样的。


In [ ]:
expected_params = {"mlp": 394634, "cnn": 136874}
for kind in ("mlp", "cnn"):
    model = build_model(kind)
    sample = x_batch[:4].to(DEVICE)
    print("\n", kind)
    with torch.no_grad():
        for layer in model:
            sample = layer(sample)
            print(f"{layer.__class__.__name__:12s} → {tuple(sample.shape)}")
    assert tuple(sample.shape) == (4, 10)
    count = sum(p.numel() for p in model.parameters())

    # !如果缺漏了某个参数段，这里会直接报错。
    assert count == expected_params[kind], (kind, count)
    print("parameters:", f"{count:,}")


## Q4 · 训练前检查

第五次实验已实现训练循环，这次复用 `lab_utils.py` 中的 `fit_model`。它仍执行：清梯度 → 前向 → 交叉熵 → 反向 → 优化器更新；每轮用同一组固定参数评估无增强训练集和验证集。

先运行下面的检查：用一批数据验证输出、损失、梯度和参数更新，再在固定 32 张训练图像上重复学习。


In [ ]:
check_model = build_model("cnn")
check_x, check_y = x_batch[:32].to(DEVICE), y_batch[:32].to(DEVICE)
criterion = nn.CrossEntropyLoss()
check_optimizer = torch.optim.Adam(check_model.parameters(), lr=0.003)
before = snapshot(check_model)
check_optimizer.zero_grad(set_to_none=True)
scores = check_model(check_x)
initial_loss = criterion(scores, check_y)
assert scores.shape == (32, 10) and torch.isfinite(initial_loss)
initial_loss.backward()
assert all(
    p.grad is not None and torch.isfinite(p.grad).all()
    for p in check_model.parameters()
)
assert all(torch.equal(before[k], v.cpu()) for k, v in check_model.state_dict().items())
check_optimizer.step()
assert any(
    not torch.equal(before[k], v.cpu()) for k, v in check_model.state_dict().items()
)

for _ in range(99):
    check_optimizer.zero_grad(set_to_none=True)
    loss = criterion(check_model(check_x), check_y)
    loss.backward()
    check_optimizer.step()
check_model.eval()
with torch.no_grad():
    tiny_scores = check_model(check_x)
    tiny_loss = criterion(tiny_scores, check_y).item()
    tiny_accuracy = (tiny_scores.argmax(1) == check_y).float().mean().item()
print(
    f"32 training images, 100 updates: loss {initial_loss.item():.4f} → {tiny_loss:.4f}, "
    f"accuracy={tiny_accuracy:.2%}"
)
assert tiny_loss < initial_loss.item(), (
    "未降低损失时，先检查模型、学习率和数据，再完整训练"
)
del check_model, check_optimizer


## Q5 · CNN 与全连接网络的比较

两组使用相同的 RGB 图像、划分、预处理、批次顺序、优化器、学习率和轮数，分别从头初始化。`fit_model` 会保存最佳验证轮次的独立参数副本，并在返回前恢复它。

曲线包含第 0 轮，即未训练时的表现。训练曲线来自无增强训练视图的重新评估，与验证曲线对应同一组固定参数；训练过程中边更新边计算的损失另外记录为 `online_loss`。

**先预测：** 哪种模型的验证表现可能更好？参数更少的 CNN 会更快吗？


In [ ]:
def run_experiment(kind, augment=False, training_view=None):
    model = build_model(kind, SEED)  # 每组重新初始化；增强对照使用相同 CNN 初值。
    if training_view is None:
        training_view = with_augmentation(train_set) if augment else train_set
    assert training_view.indices == train_set.indices
    name = f"{kind}_aug" if augment else kind
    result = fit_model(
        model,
        training_view,
        train_eval_set,
        val_set,
        device=DEVICE,
        epochs=EPOCHS,
        lr=LR,
        batch_size=BATCH_SIZE,
        order_seed=ORDER_SEED,
        name=name,
    )
    result.update(
        kind=kind, augment=augment, seed=SEED, split_seed=SPLIT_SEED, mode=MODE
    )
    return result


results = {}
for kind in ("mlp", "cnn"):
    results[kind] = run_experiment(kind)
print_comparison(results)
_ = plot_history(results)


记录两种模型的参数量、最佳验证轮次、验证准确率、验证损失和耗时。

**问题：** 哪种方案更值得继续尝试？结果是否支持你的预测？相同轮数是否意味着相同计算量？


## Q6 · 受控改进：数据增强

真实物体可能出现在不同位置、朝向不同。训练时随机变换图像，可以让模型见到同一对象的不同外观，但变换应尽可能保持标签含义。

本次比较 **CNN 无增强** 与 **CNN 有增强**：后者使用“反射填充 4 像素后的随机 32×32 裁剪 + 概率 0.5 的水平翻转”，再使用相同的张量转换和标准化。验证与测试不做随机增强。

**先预测：** 增强后的训练任务会更难还是更容易？验证准确率一定会上升吗？为什么不直接沿用 MNIST 的数字翻转规则？


In [ ]:
augmented_train = with_augmentation(train_set)
assert augmented_train.indices == train_set.indices
assert augmented_train.base is train_set.base
# fork_rng 使演示消耗的随机数不会影响后续训练。
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(7)
    augmented_images = torch.stack([augmented_train[7][7] for _ in range(6)])
_ = plot_samples(
    augmented_images,
    torch.full((6,), int(y_batch[0])),
    title="Same training image, random augmentation",
)


In [ ]:
# TODO：取得增强训练视图；需要与原训练视图使用完全相同的原始样本索引。
augmentation_view = ...
assert augmentation_view.indices == train_set.indices
# 显式核验：验证视图仍是确定性的。
assert torch.equal(val_set[0][0], val_set[0][0])
results["cnn_aug"] = run_experiment(
    "cnn", augment=True, training_view=augmentation_view
)
print_comparison(results)
_ = plot_history({k: results[k] for k in ("cnn", "cnn_aug")})


**问题：** 增强是否改善了本次验证表现？如果没有改善，可以直接判断“数据增强无效”吗？再提出一种你愿意验证的改进，说明控制条件和评价方法。


**回答与观察：** TODO（先写预期，运行后再补充证据。）


## Q7 · 最终测试与错误分析

现在根据三组的最佳验证表现选择一个方案。选择完成后，才加载官方测试集，在一次评估中得到总体指标、分类别表现、混淆矩阵和前 12 张错分图像。

混淆矩阵的行是真实类别，列是预测类别；每行归一化后表示“该真实类别被分到各类的比例”。观察不对称的混淆，不要只看对角线。


In [ ]:
# Python max 在完全相同时保留字典中较早插入的方案；此处先按准确率，再按损失。
selected_name = max(
    results,
    key=lambda k: (results[k]["best_val_accuracy"], -results[k]["best_val_loss"]),
)
selected = results[selected_name]
print("Frozen selection:", selected_name, "best epoch:", selected["best_epoch"])
test_set = load_test(download=DOWNLOAD)
test_report = evaluate(
    selected["model"], make_loader(test_set, BATCH_SIZE), DEVICE, details=True
)
print(
    f"test n={test_report['n']}, accuracy={test_report['accuracy']:.2%}, "
    f"loss={test_report['loss']:.4f}"
)
confusion = test_report["confusion"]
for i, name in enumerate(CLASSES):
    total = int(confusion[i].sum())
    rate = float(confusion[i, i]) / total if total else float("nan")
    print(f"{name:12s}: {rate:.2%} ({int(confusion[i, i])}/{total})")
_ = plot_confusion(confusion)
errors = test_report["errors"]
if errors:
    _ = plot_samples(
        torch.stack([e[0] for e in errors]),
        [e[1] for e in errors],
        predictions=[e[2] for e in errors],
        probabilities=[e[3] for e in errors],
        title="First test errors (fixed data order)",
    )
else:
    print("本次未发现错分样本。")

# 保存足以重建本实验推理模型的参数与预处理约定；不包含优化器状态，不能精确续训。
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
torch.save(
    {
        "state_dict": snapshot(selected["model"]),
        "kind": selected["kind"],
        "classes": list(CLASSES),
        "mean": [0.5] * 3,
        "std": [0.5] * 3,
        "mode": MODE,
        "best_epoch": selected["best_epoch"],
        "train_n": len(train_set),
        "split_seed": SPLIT_SEED,
        "seed": SEED,
        "order_seed": ORDER_SEED,
        "augment": selected["augment"],
        "epochs": EPOCHS,
        "lr": LR,
        "batch_size": BATCH_SIZE,
    },
    output_dir / "best_cifar10.pt",
)
print("Saved:", output_dir / "best_cifar10.pt")


**写一段实验结论，至少回答：**

1. 与 MLP 相比，本次 CNN 的验证表现、参数量和运行时间如何？
2. 数据增强的结果是否符合你的预期？哪些解释还需要进一步实验？
3. 哪两类容易混淆？先描述图像或混淆矩阵中的证据，再提出可能原因。
4. 模型给出较大的 softmax 分数，是否意味着预测一定可靠？

测试集用于报告冻结方案的结果。错分图可以启发后续工作，不能据此反复选模型再继续把同一测试集称为独立评估。


**回答与观察：** TODO（先写预期，运行后再补充证据。）


## Q8 · 选做：观察第一层特征图

将同一张验证图像送入已训练 CNN 的第一层卷积与 ReLU，查看不同通道的响应。这部分不更新模型，也不读取测试数据。

**问题：** 不同通道的响应有哪些差异？只看这些图，能否断言某个通道就是“猫检测器”？


In [ ]:
RUN_FEATURE_MAPS = False  # 选做时改为 True。
if RUN_FEATURE_MAPS:
    cnn_model = results["cnn"]["model"]
    cnn_model.eval()
    image_val, label_val = val_set[0]
    with torch.no_grad():
        features = cnn_model[:2](image_val.unsqueeze(0).to(DEVICE))[0].cpu()
    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    for c, ax in enumerate(axes.flat):
        ax.imshow(features[c], cmap="viridis")
        ax.set_title(f"channel {c}")
        ax.axis("off")
    fig.suptitle(f"First-layer responses, true class: {CLASSES[label_val]}")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()


**回答与观察：** TODO（先写预期，运行后再补充证据。）


## Q9 · 选做：用 SVD 压缩图像信息

一张灰度图可以看作一个矩阵，RGB 图像则可以看作三个矩阵。能不能只保存少量数值，就近似还原整张图像？本题用奇异值分解（SVD）展示一种**有损的低秩表示**，不需要训练神经网络。

对一个颜色通道 $A\in\mathbb{R}^{H\times W}$，有

$$
A=U\Sigma V^{\mathsf T}
 =\sum_{i=1}^{r}\sigma_i u_i v_i^{\mathsf T},
\qquad r=\min(H,W),\quad \sigma_1\geq\sigma_2\geq\cdots\geq0.
$$

只保留前 $k$ 个奇异值及对应向量，得到

$$
A_k=U_{:,1:k}\operatorname{diag}(\sigma_1,\ldots,\sigma_k)V^{\mathsf T}_{1:k,:}.
$$

它是在秩不超过 $k$ 的矩阵中，使 Frobenius 重建误差最小的近似。直观上，逐项加入这些秩一矩阵，就逐步补充了图像中的结构与细节。对 RGB 三个通道分别这样处理，再将结果堆叠起来显示。

**存储量怎样算？** 原 RGB 图像有 $3HW$ 个像素数值；如果每个通道仅保存截取后的 $U_k$、长度为 $k$ 的奇异值向量和 $V_k^{\mathsf T}$，共需 $3k(H+W+1)$ 个系数。只存奇异值、丢掉左右奇异向量，不能恢复图像。

下面提供完整演示。只需先运行环境配置和 Q1 的数据加载单元，就可以跳到这里；不依赖模型训练。将 `RUN_SVD` 改为 `True` 后运行。默认使用一张训练图像；也可以把 `SVD_IMAGE_PATH` 改为自己电脑上一张照片的路径，观察更丰富的细节（代码将长边限制为 256 像素）。

**先预测：** $k$ 越大是否越清楚？完整保留全部分量时，是否还能节省存储？


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


def svd_reconstruct(parts, k):
    """parts 是三个通道的 (U, s, Vt)；每个通道只保留前 k 项。"""
    if not 1 <= k <= min(len(s) for _, s, _ in parts):
        raise ValueError("k 必须在 1 与图像的较短边长度之间")
    # U 的各列乘以对应奇异值，再做二维矩阵乘法。
    return np.stack(
        [np.dot(U[:, :k] * s[:k], Vt[:k, :]) for U, s, Vt in parts], axis=-1
    )


RUN_SVD = False  # 选做时改为 True；本题代码已完整给出。
SVD_IMAGE_PATH = None  # 例如 "/你的路径/photo.jpg"；None 表示使用训练集图像。
SVD_IMAGE_INDEX = 7

if RUN_SVD:
    if SVD_IMAGE_PATH is None:
        # Q1 的训练评估视图没有随机增强；display_image 还原到 [0,1]、HWC。
        svd_rgb = np.asarray(display_image(train_eval_set[SVD_IMAGE_INDEX][0]), dtype=np.float32)
    else:
        with Image.open(SVD_IMAGE_PATH) as svd_file:
            svd_photo = svd_file.convert("RGB")
            svd_photo.thumbnail((256, 256))
            svd_rgb = np.asarray(svd_photo, dtype=np.float32) / 255.0

    svd_h, svd_w, svd_channels = svd_rgb.shape
    svd_r = min(svd_h, svd_w)
    # full_matrices=False：只计算重建所需的较小形状；s 按从大到小排列。
    svd_parts = [np.linalg.svd(svd_rgb[:, :, c], full_matrices=False) for c in range(3)]
    svd_ranks = sorted({min(k, svd_r) for k in (1, 2, 4, 8, 16)} | {svd_r})
    svd_total_energy = sum(float(np.sum(s.astype(np.float64) ** 2)) for _, s, _ in svd_parts)
    svd_original_values = svd_channels * svd_h * svd_w
    svd_original_u8_bytes = svd_original_values  # 原图按每个通道数值 uint8、1 字节计。
    svd_rows, svd_reconstructions = [], []

    print(f"Image: {svd_h} × {svd_w} × 3; raw uint8 bytes: {svd_original_u8_bytes:,}")
    print(" k     MSE       energy    coefficients/original    factor bytes/raw uint8 bytes")
    for svd_k in svd_ranks:
        svd_rec = svd_reconstruct(svd_parts, svd_k)
        # 在裁剪前计算误差；显示时才裁剪，避免越界颜色影响绘图。
        svd_mse = float(np.mean((svd_rgb.astype(np.float64) - svd_rec) ** 2))
        svd_kept = sum(float(np.sum(s[:svd_k].astype(np.float64) ** 2)) for _, s, _ in svd_parts)
        svd_energy = svd_kept / svd_total_energy if svd_total_energy > 0 else 1.0
        svd_count = svd_channels * svd_k * (svd_h + svd_w + 1)
        # 假设只保存截取后的因子；不包含文件头、形状记录等额外开销。
        svd_factor_bytes = sum(
            U[:, :svd_k].nbytes + s[:svd_k].nbytes + Vt[:svd_k, :].nbytes
            for U, s, Vt in svd_parts
        )
        svd_rows.append({"k": svd_k, "mse": svd_mse, "energy": svd_energy,
                         "coefficient_ratio": svd_count / svd_original_values,
                         "byte_ratio": svd_factor_bytes / svd_original_u8_bytes})
        svd_reconstructions.append(svd_rec)
        print(f"{svd_k:2d}  {svd_mse:9.3e}  {svd_energy:8.2%}"
              f"          {svd_count / svd_original_values:8.2%}"
              f"                     {svd_factor_bytes / svd_original_u8_bytes:6.2f}x")

    svd_fig, svd_axes = plt.subplots(2, 4, figsize=(12, 8))
    svd_axes.flat[0].imshow(svd_rgb)
    svd_axes.flat[0].set_title("Original RGB")
    for svd_ax, svd_row, svd_rec in zip(svd_axes.flat[1:], svd_rows, svd_reconstructions):
        svd_ax.imshow(np.clip(svd_rec, 0, 1))
        svd_ax.set_title(f"k={svd_row['k']}, MSE={svd_row['mse']:.2e}\n"
                         f"coefficients={svd_row['coefficient_ratio']:.1%}", fontsize=10)
    for svd_i, svd_ax in enumerate(svd_axes.flat):
        svd_ax.axis("off")
        if svd_i > len(svd_rows):
            svd_ax.set_visible(False)
    svd_fig.suptitle("SVD: image quality and representation size")
    svd_fig.subplots_adjust(left=0.02, right=0.98, bottom=0.04, top=0.86,
                            hspace=0.45, wspace=0.12)
    plt.show()

    np.testing.assert_allclose(svd_reconstruct(svd_parts, svd_r), svd_rgb, atol=1e-5)
    print("Full-rank reconstruction check passed (allowing floating-point error).")


**观察与思考：**

1. 选择一个你认为“还能认出主体”的 $k$，记录它的重建误差和系数比例，说明哪些细节先消失了。同学之间可以有不同的视觉判断，但要给出依据。
2. 表中的 `energy` 是保留奇异值平方和占全部奇异值平方和的比例（三个通道合计）。保留较多能量，是否意味着所有有意义的细节都还在？
3. 对 32×32 的 RGB 图像，$k=8$ 时系数比例是多少？如果原图使用 uint8，而因子使用 float32，字节数是否真的减少？
4. 本题与神经网络分类有什么不同？图像在视觉上还能辨认，是否就能保证分类模型仍然预测正确？

**说明：** `coefficients/original` 只比较数值个数，等价于假设两边使用相同数值精度；`factor bytes/raw uint8 bytes` 进一步计入因子的实际 dtype。两者都不是与 JPEG、PNG 文件大小的比较。本题只演示低秩表示，没有生成压缩文件；如果要在磁盘上只保存 $k$ 项，需要保存截取后的三个因子及必要形状信息，而不是保存完整分解或完整重建图像。

图中为显示而做了 [0,1] 裁剪；MSE 在裁剪前计算。保留全部分量时误差应接近零，但分解后的因子存储量可能反而更大。

接口参考：[NumPy：numpy.linalg.svd](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html)。


**回答与观察：** 选做时记录你选择的 k、误差、存储量与判断依据。


## 提交与自查

提交可从头运行的 notebook，保留模型、形状与参数检查、三组比较结果、曲线、最终测试和你的解释。标明 quick/full、设备、训练规模和轮数；使用 AI 时，简述一条关键建议及你如何核验。

提交前检查：是否只在训练视图中使用随机增强？三组是否使用同一划分与预算？是否先看验证结果再做最终测试？结论是否区分观察、解释和待验证的猜测？

## 参考资料

- [CIFAR-10 数据集说明](https://cave.cs.toronto.edu/kriz/cifar.html)：图像、类别与官方训练／测试划分。
- [d2l：从全连接层到卷积](https://zh.d2l.ai/chapter_convolutional-neural-networks/why-conv.html)：局部性与共享权重。
- [d2l：图像卷积](https://zh.d2l.ai/chapter_convolutional-neural-networks/conv-layer.html)：局部计算与图像滤波。
- [d2l：卷积神经网络（LeNet）](https://zh.d2l.ai/chapter_convolutional-neural-networks/lenet.html)：卷积块与分类层的组织方式。
- [PyTorch：Training a Classifier](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)：CIFAR-10 彩图分类流程。

本实验的任务与问题为课程自拟。采用独立验证集，不按每轮测试集结果选模型；数据增强、CPU 子集与对照安排也与原教程不同。
